## A/B Testing

A/B Testing (or split testing) is a controlled experiment that compares two versions (A and B) of a single variable, such as a webpage element, product, feature, or business strategy, to determine which one maximizes a defined outcome (the conversion rate). The goal is to use statistical hypothesis testing to decide whether the performance difference between A and B is real (statistically significant) or just due to random chance. Common use cases include testing website layouts, email campaigns, product designs, or call-to-action wording to optimize business metrics like conversion rates or sales. A/B testing helps eliminate guesswork by enabling data-driven decisions in user experience optimization.

A/B testing starts with defining a measurable goal aligned with business objectives, like increasing conversion rate or sales through a landing page. With a goal set, create a hypothesis for how to optimize the page and drive improvement. Conduct user research to inform the hypotheses. This is often done by dividing users into two distincg groups: control and treatment group.

* **Control Group**: This group is shown the existing page (version A).

* **Treatment Group**: This group is shown the new proposed page (version B).

### The A/B Testing Cycle

1. **Hypothesize**: Define the expected outcome and set the null/alternative hypotheses.

2. **Set up the Experiment**: Determine the required sample size and run the test.

3. **Analyze the Results**: Perform the hypothesis test to calculate the P-value.

4. **Decision and Implementation**: Conclude whether to reject the null hypothesis and implement the winning version.

## Setting up the Experiment

Once we have defined the expected outcome and set the null/alternative hypothesis. The next step is to determine the sample size and how long we should run the experiment for.

### Determing the sample size

Running a test without enough data risks a Type II error (failing to detect a real difference). Sample size calculation ensures the test has enough statistical power (typically 80%) to reject the null hypothesis. Determining the sample size depends on the following factors:

* **Baseline conversion rate** (current performance of the control group)

* **Minimum Detectable Effect (MDE)**: the smallest effect size you want to detect (e.g., a 5% increase in conversion)

* **Statistical significance level** (usually set at 95% confidence)

* **Statistical power** (usually set at 80%, the likelihood of detecting an effect if it exists)

Let's see how to calculathe the sample size with python using `statmodels` library.

**Example 1**: Suppose a company wants to test a new call-to-action button color. The current (baseline) conversion rate ($p_A$) is 4%. The page receives 5,000 visitors per day. The change is only worth implementing if it achieves at least a 20% relative increase in conversion rate.

In [1]:
from statsmodels.stats.power import zt_ind_solve_power
from statsmodels.stats.proportion import proportion_effectsize
import numpy as np

# --- Define Parameters (from Scenario) ---
baseline_rate = 0.04  # p_A
mde_absolute = 0.008  # p_B - p_A (0.2 * baseline_rate)
alpha = 0.05  # Significance Level
power = 0.80  # Statistical Power

# Calculate Cohen's h (Effect Size for Proportions)
effect_size = proportion_effectsize(
    prop1=baseline_rate, prop2=baseline_rate + mde_absolute
)

# --- Calculate required 'n' per group ---
required_n_per_group = zt_ind_solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    alternative="two-sided",
    ratio=1.0,  # Assumes n_A = n_B
)

N_REQUIRED = int(np.ceil(required_n_per_group))

print(f"Baseline Rate: {baseline_rate * 100:.2f}%")
print(f"Target Rate (p_B): {(baseline_rate + mde_absolute) * 100:.2f}%")
print(f"Required Sample Size PER GROUP (n): {N_REQUIRED}")
print(f"Total Sample Size Required: {N_REQUIRED * 2}")

Baseline Rate: 4.00%
Target Rate (p_B): 4.80%
Required Sample Size PER GROUP (n): 10297
Total Sample Size Required: 20594


[Cohen's h](http://en.wikipedia.org/wiki/Cohen%27s_h) effect size is defined as the difference between the arcsine transformation of the proportions we are trying to test for. For difference in means, we would use [Cohen's d](https://en.wikipedia.org/wiki/Effect_size#Cohen's_d) effect size, defined as the difference between the means divided by the pooled standard deviation where $n_1=n_2=0$.

$$ d = \frac{|{\bar{x}_1 - \bar{x}_2}|}{s_p}, \ \text{where} \ sp = \sqrt{\frac{s_1^2 + s_2^2}{2}}$$

If our expected outcome is an increase the average value of a metric (e.g. revenue per user), then we would use the `statsmodels.stats.power.tt_ind_solve_power` function to determine the number of samples.

### Determining the test duration

Here we use the total required sample size and the daily traffic figures to find the minimum number of days the experiment must run. The formula roughly is:

$$ \text{Test Duration (Days)} = \frac{\text{Required Sample Size}}{\text{Daily Visitors}} $$

It is recommended to runn the test for at least 1-2 weeks to cover typical behavior cycles and ensure stable results. Ending an A/B test too early risks misleading conclusions.

In [2]:
# --- Calculate Test Duration (from Scenario) ---
daily_visitors = 5000     # 5,000 visitors per day
total_n_required = N_REQUIRED * 2 # Total samples required

test_duration_days = total_n_required / daily_visitors

# Ensure the duration is at least one full week (7 days) to account for day-of-week effects
duration = max(7, np.ceil(test_duration_days))

print(f"\nDaily Visitors: {daily_visitors}")
print(f"Total N Required: {total_n_required}")
print(f"Calculated Test Duration: {duration:.0f} days (Minimum 7 days)")


Daily Visitors: 5000
Total N Required: 20594
Calculated Test Duration: 7 days (Minimum 7 days)


Now we run the experiment for 7 days, show version A of the app the half of the daily visitors and version B to the remaining half.

## Analysing the Result (Hypothesis Testing)

The next step after the running the experiment for the desired duration is to then analyze the result. Suppose the result for the experiment is given in the table below.

| Group | Visitors (n) | Conversions (Successes) |
| --- | --- | --- |
| Control (Version A) | 17,500 | 1,750 |
| Treatment (Version B) | 17,500 | 1,950 |

First we need to formulate the hypothesis. We assume the two groups' conversion rates, $p_A$ and $p_B$, are the same until proven otherwise.

* Null Hypothesis ($H_0$): There is no difference in conversion rate.
    $$ H_0: p_A = p_B $$

* Alternative Hypothesis ($H_1$): There is a difference in conversion rates.
    $$ H_1: p_A \ne p_B$$

We'll use a significance level ($\alpha$): 0.05

In [24]:
# Raw Data
n_A = 17500
conv_A = 1750

n_B = 17500
conv_B = 1950

# Sample Proportions
p_A = conv_A / n_A
p_B = conv_B / n_B

print(f"Control Conversion Rate (p_A): {p_A:.4f}")
print(f"Treatment Conversion Rate (p_B): {p_B:.4f}")
print(f"Observed Difference: {p_B - p_A:.4f}")

Control Conversion Rate (p_A): 0.1000
Treatment Conversion Rate (p_B): 0.1114
Observed Difference: 0.0114


For this experiment, the suitable test is a two-sample z-test for proportion.

In [25]:
from scipy import stats

# 1. Calculate the Pooled Proportion (p_c)
total_conversions = conv_A + conv_B
total_visitors = n_A + n_B
p_c = total_conversions / total_visitors

# 2. Calculate the Standard Error (SE) using p_c
SE_prop = np.sqrt(p_c * (1 - p_c) * (1/n_A + 1/n_B))

# 3. Calculate the Z-statistic
Z_stat = (p_B - p_A) / SE_prop

# 4. Calculate the P-value (Two-tailed)
p_value = 2 * (1 - stats.norm.cdf(abs(Z_stat)))

print("--- Two-Sample Z-Test Results ---")
print(f"Pooled Proportion (p_c): {p_c:.4f}")
print(f"Calculated Z-statistic: {Z_stat:.4f}")
print(f"P-value (Two-tailed): {p_value:.4f}")

--- Two-Sample Z-Test Results ---
Pooled Proportion (p_c): 0.1057
Calculated Z-statistic: 3.4769
P-value (Two-tailed): 0.0005


Alternatively, we can use `statsmodels.stats.proportions.proportions_ztest` for calculate the test statistic and p-value.

In [5]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

counts = [conv_B, conv_A]
nobs = [n_B, n_A]

alt_Z_stat, alt_p_value = proportions_ztest(counts, nobs)

print("--- Two-Sample Z-Test Results ---")
print(f"Calculated Z-statistic (Alternative): {alt_Z_stat:.4f}")
print(f"P-value (Two-tailed) (Alternative): {alt_p_value:.4f}")

--- Two-Sample Z-Test Results ---
Calculated Z-statistic (Alternative): 3.4769
P-value (Two-tailed) (Alternative): 0.0005


## Interpretation and Decision

Now that we have our p-value, we compare is the chosen significance level, $alpha=0.05$.

In [6]:
alpha = 0.05

if p_value < alpha:
    print("Decision: Reject the Null Hypothesis.")
    print("Conclusion: The conversion rate of Treatment (B) is significantly different from Control (A).")
else:
    print("Decision: Fail to Reject the Null Hypothesis.")
    print("Conclusion: There is no significant difference between the two versions.")

Decision: Reject the Null Hypothesis.
Conclusion: The conversion rate of Treatment (B) is significantly different from Control (A).


We could also compute the confidence inverval (CI) for the difference in proportion to see the range of possible differences between $p_B$ and $p_A$.

In [7]:
from statsmodels.stats.proportion import confint_proportions_2indep

CI_lower, CI_upper = confint_proportions_2indep(conv_B, n_B, conv_A, n_A, alpha=0.05)

print("\n--- 95% Confidence Interval for Difference in Proportion---")
print(f"95% CI: ({CI_lower:.4f}, {CI_upper:.4f})")


--- 95% Confidence Interval for Difference in Proportion---
95% CI: (0.0050, 0.0179)


Since the entire 95% CI range (0.0050 to 0.0179) is above zero, we confirm that the difference is statistically significant, and we are 95% confident that the true lift in conversion rate is between 0.50 and 1.79 percentage points.

Another type of question we might be interested in is, "Does changing the button influence conversion?". In this case, the suitable test for the analysis would be chi-square test for independence. The update table for the results is going to be as follows:

| Group | Visitors (n) | Converted | No Converted |
| --- | --- | --- | --- |
| Control (Version A) | 17,500 | 1,750 | 15,750 |
| Treatment (Version B) | 17,500 | 1,950 | 15,550 |

Null and alternative hypothesis:

* $H_0$: Conversion is independent of button colour
* $H_1$: Conversion is dependent on button colour

In [8]:
from scipy.stats import chi2_contingency
import numpy as np

# Observed Frequencies (2x2 Contingency Table)
# Rows: Group (A, B)
# Columns: Outcome (Converted, Not Converted)
observed_data = np.array(
    [
        [1750, 15750],  # Control
        [1950, 15500],  # Treatment
    ]
)

alpha = 0.05

# Perform the Chi-square Test for Independence
chi2_stat, p_value_chi2, degrees_of_freedom, expected_data = chi2_contingency(
    observed_data
)

print("--- Chi-square Test Results ---")
print(f"Calculated Chi-square Statistic: {chi2_stat:.4f}")
print(f"Degrees of Freedom: {degrees_of_freedom}")
print(f"P-value: {p_value_chi2:.4f}")

if p_value_chi2 < alpha:
    print(
        "Conclusion: Reject H0. Conversion is dependent on call-to-action button colour."
    )
else:
    print(
        "Conclusion: Fail to Reject H0. Conversion is independent of call-to-action button colour."
    )

--- Chi-square Test Results ---
Calculated Chi-square Statistic: 12.6155
Degrees of Freedom: 1
P-value: 0.0004
Conclusion: Reject H0. Conversion is dependent on call-to-action button colour.


Let's see another example, this time comparing means. Suppose a retail company tests a new product recommendation layout (Version B) to see if it increases the Average Order Value (AOV) and got the following result:

| Group | Sample Size | Sample Mean | Sample Standard Deviation |
| :--- | :--- | :--- | :--- |
| Control | 1,200 | \$52.50 | \$15.00 |
| Treatement | 1,180 | \$54.10 | \$16.50 |

The null and alternative hypothesis:

$$ H_0: \mu_A = \mu_B \ \text{(The average order values are equal)} $$
$$ H_1: \mu_A \ne \mu_B \ \text{(The average order values are different)} $$

Since we only have the sample statistics (n and s) and the underlying population standard deviation (\sigma) for AOV is unknown, the T-test is appropriate. We assume unequal variances (Welch's test).

In [9]:
from scipy.stats import ttest_ind_from_stats

# Data for AOV Comparison
n_A_aov = 1200
mean_A_aov = 52.50
std_A_aov = 15.00

n_B_aov = 1180
mean_B_aov = 54.10
std_B_aov = 16.50

alpha = 0.05

# Perform the T-test using sample statistics (Welch's test: equal_var=False)
t_stat_aov, p_value_aov = ttest_ind_from_stats(
    mean1=mean_B_aov,
    std1=std_B_aov,
    nobs1=n_B_aov,
    mean2=mean_A_aov,
    std2=std_A_aov,
    nobs2=n_A_aov,
    equal_var=False,
)

print("\n--- Two-Sample T-Test (AOV Comparison) ---")
print(f"Calculated T-statistic: {t_stat_aov:.4f}")
print(f"P-value (Two-tailed): {p_value_aov:.4f}")

if p_value_aov < alpha:
    print("Conclusion: Reject H0. The new layout significantly changed the AOV.")
else:
    print("Conclusion: Fail to Reject H0. No significant difference in AOV found.")


--- Two-Sample T-Test (AOV Comparison) ---
Calculated T-statistic: 2.4741
P-value (Two-tailed): 0.0134
Conclusion: Reject H0. The new layout significantly changed the AOV.


## Common A/B Test Pitfalls

While A/B testing is a powerful tool, it's easy to make mistakes that invalidate our results. These errors often stem from ignoring statistical principles or failing to account for user behavior.

* **Peeking**: Checking results before the predetermined sample size is reached. This artificially inflates the Type I error rate.

* **Novelty Effect**: Changes sometimes cause a short-term spike in engagement simply because they are new, not because they are inherently better. Always ensure the test runs long enough to account for weekly cycles.

* **Testing Multiple Variables at Once**: Changing more than one element simultaneously makes it impossible to isolate which change drove any observed effect. Testing one variable at a time ensures clear interpretation of results.

* **Unequal or Inconsistent Audience Exposure**: Exposing variations to different user segments or demographics distorts comparisons. Test groups should be as similar as possible in user characteristics.

* **Misinterpreting Statistical Significance**: Relying solely on p-values without considering practical significance or effect size can lead to misguided decisions. It's important to evaluate the magnitude and business impact of differences.

# Exercises

1. A streaming company ran an A/B test for 14 days to see if moving their "Subscribe Now" button from the bottom of the page (Control, A) to the top (Treatment, B) would increase sign-ups.

    | Group | Visitors ($n$) | Sign-ups (Conversions) |
    | :--- | :--- | :--- |
    | **A (Control)** | 10,000 | 250 |
    | **B (Treatment)** | 10,000 | 290 |

    Using a significance level of $\alpha = 0.05$, answer the following questions:

    i.  What is the Null and Alternative Hypotheses ($H_0$ and $H_1$)?

    ii.  What is the pooled conversion rate ($\hat{p}_c$) and the $Z$-statistic?

    iii.  What is the $P$-value?

    iv. Is the change statistically significant?

    v. What is your final conclusion?

    **Hint:** Use the **Two-Sample Z-test for Proportions**. For Python, the `statsmodels.stats.proportion.proportions_ztest` function is useful, or use `scipy.stats.norm.cdf` to find the $P$-value manually.


2. A company wants to test a new checkout feature (Treatment) to see if it significantly changes the **Average Revenue Per User (ARPU)**. The new feature is only worth launching if it increases the current ARPU by at least **\$0.75**.

    **Given Parameters:**

    * Baseline ARPU ($\mu_A$): **\$15.50**
    * Baseline Standard Deviation ($\sigma_A$): **\$5.00**
    * Target Power ($1-\beta$): **$0.80$**
    * Significance Level ($\alpha$): **$0.05$**
    * Average Daily Users (to the test section): **$2,500$**

    Using **Cohen's $d$**, assuming $\sigma_A = \sigma_B$. Calculate the minimum required sample size and the minimum duration of the test in days.

    **Hint:** Use the Python function for power analysis, **`statsmodels.stats.power.tt_ind_solve_power`**. Get the effect size using cohen's d formula.

3. An e-commerce site redesigns its checkout process (Version B) hoping to increase the **Average Order Value (AOV)**, a continuous metric. After running the experiment, the following results were obtained from the customers who made a purchase:

    | Group | Sample Size ($n$) | Sample Mean ($\bar{x}$) | Sample Standard Deviation ($s$) |
    | :--- | :--- | :--- | :--- |
    | **A (Control)** | 850 | \$45.20 | \$12.50 |
    | **B (Treatment)** | 845 | \$47.00 | \$13.80 |

    Using $\alpha = 0.05$, is the increase in AOV statistically significant?

    **Hint:** Use the **Two-Sample T-test** (Welch's test is recommended since population $\sigma$ is unknown). Use the function **`scipy.stats.ttest_ind_from_stats`** setting `equal_var=False`.

# Q3 Answer
#### Q1i

In [10]:
# Given:

# H0: mean_a >= mean_b
# H1: mean_a < mean_b

n_aov_a = 850
sample_mean_a = 45.20
sample_std_a = 12.50

n_aov_b =845
sample_mean_b = 47.00
sample_std_b = 13.80

alpha = 0.05

t_stat_aov, p_value_aov = ttest_ind_from_stats(
    mean1=sample_mean_a,
    std1=sample_std_a,
    nobs1=n_aov_a,
    mean2=sample_mean_b,
    std2=sample_std_b,
    nobs2=n_aov_b,
    alternative="less",
    equal_var=False,
)

print(f"T_statistic: {t_stat_aov}")
print(f"P_value: {p_value_aov}")

print ("---Conclusion---")
if p_value_aov < alpha:
    print ("Reject H0: Moving the button uppward would make a significant change on the channel")
else: ("Fail to reject H0: No significant cahnge was found on the channel")

T_statistic: -2.813886983299315
P_value: 0.0024759197356251407
---Conclusion---
Reject H0: Moving the button uppward would make a significant change on the channel


# Q2 Answer

In [37]:
from statsmodels.stats.power import tt_ind_solve_power

In [52]:
# mean_a(Ho) <= mean_b(Ha)
# mean_b(Ha) > mean_a(Ho)
# Given:
baseline_arpu = 15.50
bsd = 5.00
mde = 0.75  # Minimum Detectable Effect
expected_arpu = baseline_arpu + mde
alpha = 0.05
power = 0.80
daily_visitors = 2500

# Calculate the effect size
effect_size = abs(baseline_arpu - expected_arpu)/bsd

N_samples_per_group = tt_ind_solve_power(
    effect_size = effect_size,
    alpha=alpha,
    power=power,
    alternative="larger"
)

# Convert to int and round up

N_samples_per_group = int(np.ceil(N_samples_per_group))
total_n_required = N_samples_per_group * 2

test_duration_days = total_n_required / daily_visitors
duration = max(7, np.ceil(test_duration_days))


In [51]:
test_duration_days

0.5592

# Q1 Answer

In [29]:
import numpy as np
from scipy.stats import norm
from statsmodels.stats.proportion import proportions_ztest

# H0: p_B = p_A        (No difference in conversion rates)
# H1: p_B > p_A        (Button at top INCREASES conversions)

visitors_A = 10000
visitors_B = 10000
mean_A = 250
mean_B = 290

alpha = 0.05 

# Sample proportions
p_A = mean_A / visitors_A
p_B = mean_B / visitors_B

# Pooled conversion rate
Pc = (mean_A + mean_B) / (visitors_A + visitors_B)

# Standard error
se = np.sqrt(Pc * (1 - Pc) * (1/visitors_A + 1/visitors_B))

# Z-statistic
z_stat = (p_B - p_A) / se


# iii. p-value
p_value = 1 - norm.cdf(z_stat)

print("---- RESULTS ----")
print(f"Conversion A: {p_A:.4f}")
print(f"Conversion B: {p_B:.4f}")
print(f"Pooled rate: {Pc:.4f}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")


print("\n --Conclusion--:")

if p_value < alpha:
    print("The change is statistically significant — button at top INCREASES sign-ups.")
else:
    print("No significant increase.")

---- RESULTS ----
Conversion A: 0.0250
Conversion B: 0.0290
Pooled rate: 0.0270
Z-statistic: 1.7450
P-value: 0.040488

 --Conclusion--:
The change is statistically significant — button at top INCREASES sign-ups.
